# QuRe training on Kaggle (12-hour session)

Run with **GPU enabled** and **Internet enabled** in Kaggle Notebook settings. Set the dataset link and choose `fashionIQ` or `cirr` below. Outputs/checkpoints go to `/kaggle/working/qure_runs`; use **Save Version** with output enabled before the session ends.

This repo trains with in-batch negatives: other targets in the batch are negatives for the contrastive loss. It does not pass a separate negative image through the training batch. The current training entrypoint also runs validation each epoch; FashionIQ validation reports Recall, while CIRR validation reports Recall and subset Recall.

In [ ]:
# Cell 1 — user settings
DATASET = 'fashionIQ'  # 'fashionIQ' or 'cirr'
DRIVE_URL = 'https://drive.google.com/drive/u/0/folders/10ga17DrAZsCahRT3D3ELnzWQrAtoTK8l'
# Set this only when resuming from an earlier Kaggle output/checkpoint.
RESUME_CHECKPOINT = ''  # e.g. '/kaggle/input/qure-run/model.pth'
# First use a small number of epochs to estimate time/epoch, then increase while fitting the 12h limit.
EPOCHS = 5
BATCH_SIZE = 16
NUM_WORKERS = 4
EVAL_AFTER_TRAIN = True
assert DATASET in {'fashionIQ', 'cirr'}
assert '10ga17DrAZsCahRT3D3ELnzWQrAtoTK8l' in DRIVE_URL

In [ ]:
# Cell 2 — verify Kaggle GPU and install the project dependencies
import torch, os, sys, subprocess, time, json, shutil, zipfile
print('Python:', sys.version)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator in Kaggle Notebook settings, then rerun.')
# Kaggle currently starts this notebook on Python 3.13. transformers 4.26 pins an old tokenizers
# release that has no compatible wheel there; use a newer compatible pair instead.
# Run pip through this notebook's active Python so installation and imports use the same environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=tokenizers',
    'gdown', 'timm==0.4.12', 'transformers==4.49.0', 'tokenizers==0.21.4',
    'fairscale==0.4.4', 'iopath==0.1.10', 'webdataset==0.2.86',
    'omegaconf==2.3.0', 'pycocoevalcap==1.2', 'decord==0.6.0'], check=True)
import transformers, tokenizers
print('Transformers:', transformers.__version__, '| tokenizers:', tokenizers.__version__)
torch.backends.cudnn.benchmark = True
print(torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 — get QuRe source
REPO = '/kaggle/working/QuRe'
BRANCH = 'new_3Qformer'
if not os.path.isdir(REPO):
    !git clone --depth 1 --single-branch --branch new_3Qformer https://github.com/fast2104/QuRe.git $REPO
%cd /kaggle/working/QuRe
import subprocess
actual_branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=REPO, text=True).strip()
print('Repo ready:', os.getcwd(), '| branch:', actual_branch)
assert actual_branch == BRANCH, f'Expected {BRANCH}, got {actual_branch}'
# QuRe is image/text-only. Skip LAVIS's unrelated ULIP point-cloud processor import,
# which otherwise pulls in the very large optional Open3D package during eager imports.
from pathlib import Path
processors_init = Path(REPO)/'lavis/processors/__init__.py'
processor_text = processors_init.read_text(encoding='utf-8')
processor_text = processor_text.replace('from lavis.processors.ulip_processors import ULIPPCProcessor\n', '')
processor_text = processor_text.replace('    "ULIPPCProcessor",\n', '')
processors_init.write_text(processor_text, encoding='utf-8')
# The vendored LAVIS BLIP-Diffusion class imports a private Transformers helper
# removed in the Python-3.13-compatible Transformers version; QuRe does not use it.
models_init = Path(REPO)/'lavis/models/__init__.py'
model_text = models_init.read_text(encoding='utf-8')
model_text = model_text.replace('from lavis.models.blip_diffusion_models.blip_diffusion import BlipDiffusion\n', '')
model_text = model_text.replace('    "BlipDiffusion",\n', '')
models_init.write_text(model_text, encoding='utf-8')
# Patch old vendored LAVIS imports against current Transformers APIs in every affected module.
import re
compat_path = Path(REPO)/'lavis/models/_hf_compat.py'
compat_path.write_text("""import torch\n\ndef find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):\n    mask = torch.ones(n_heads, head_size)\n    heads = set(heads) - already_pruned_heads\n    for head in heads:\n        head -= sum(1 for pruned in already_pruned_heads if pruned < head)\n        mask[head] = 0\n    keep = mask.view(-1).contiguous().eq(1)\n    return heads, torch.arange(len(keep))[keep].long()\n""", encoding='utf-8')
helper_import = 'from lavis.models._hf_compat import find_pruneable_heads_and_indices\n'
old_helpers = re.compile(r'from transformers\.modeling_utils import \(\s*PreTrainedModel,\s*apply_chunking_to_forward,\s*find_pruneable_heads_and_indices,\s*prune_linear_layer,\s*\)', re.MULTILINE)
new_helpers = 'from transformers.pytorch_utils import (\n    apply_chunking_to_forward,\n    prune_linear_layer,\n)\nfrom transformers.modeling_utils import PreTrainedModel\n' + helper_import.rstrip()
for rel_path in ['lavis/models/med.py', 'lavis/models/blip_models/nlvr_encoder.py', 'lavis/models/blip2_models/Qformer.py']:
    target = Path(REPO)/rel_path
    content = target.read_text(encoding='utf-8')
    if 'from transformers.pytorch_utils import (' not in content:
        content, n = old_helpers.subn(new_helpers, content, count=1)
        if n:
            target.write_text(content, encoding='utf-8')
print('Patched old LAVIS Transformers imports where present')
# Catch additional vendored modules (for example modeling_t5.py) that import the removed pruning helper.
for target in (Path(REPO)/'lavis/models').rglob('*.py'):
    content = target.read_text(encoding='utf-8')
    if 'find_pruneable_heads_and_indices' not in content:
        continue
    content = content.replace('    find_pruneable_heads_and_indices,\n', '')
    if 'from lavis.models._hf_compat import find_pruneable_heads_and_indices' not in content and 'def find_pruneable_heads_and_indices(' not in content:
        match = re.search(r'(?m)^from transformers\.(?:modeling_utils|pytorch_utils) import', content)
        if match:
            content = content[:match.start()] + helper_import + content[match.start():]
        else:
            content = helper_import + content
    target.write_text(content, encoding='utf-8')
print('Checked all vendored LAVIS model modules for the removed pruning helper')
print('Skipped unused ULIP/Open3D and BLIP-Diffusion imports')

In [ ]:
# Cell 4 — download the shared Drive folder into temporary Kaggle storage
# A public Drive folder link is required. Large folders may take time; Kaggle /working is reserved for results.
import gdown, pathlib
DATA_DOWNLOAD = '/kaggle/temp/qure_dataset_download'
os.makedirs(DATA_DOWNLOAD, exist_ok=True)
downloaded = gdown.download_folder(url=DRIVE_URL, output=DATA_DOWNLOAD, quiet=False, remaining_ok=True)
print('Downloaded entries:', len(downloaded or []))
# Unpack archives if the shared folder contains .zip files.
for root, dirs, files in os.walk(DATA_DOWNLOAD):
    for filename in files:
        if filename.lower().endswith('.zip'):
            zpath = os.path.join(root, filename)
            dest = os.path.join(DATA_DOWNLOAD, os.path.splitext(filename)[0])
            if not os.path.isdir(dest) or not os.listdir(dest):
                print('Extracting', zpath)
                with zipfile.ZipFile(zpath) as zf: zf.extractall(dest)
print('Dataset download/extraction finished')

In [ ]:
# Cell 5 — locate dataset root, patch the repo's machine-specific dataset paths, and sanity-check files
from pathlib import Path
root = Path(DATA_DOWNLOAD)
if DATASET == 'fashionIQ':
    required = lambda p: (p/'captions'/'cap.dress.train.json').is_file() and (p/'images').is_dir() and (p/'image_splits').is_dir()
else:
    required = lambda p: (p/'cirr'/'captions'/'cap.rc2.train.json').is_file() and (p/'cirr'/'image_splits').is_dir()
matches = []
for walk_root, walk_dirs, walk_files in os.walk(root):
    candidate = Path(walk_root)
    if required(candidate):
        matches.append(candidate)
        break
if not matches:
    print('Top-level contents:', list(root.iterdir())[:30])
    raise FileNotFoundError('Could not find expected dataset layout. FashionIQ expects captions/, image_splits/, images/; CIRR expects cirr/captions/ and cirr/image_splits/.')
DATA_ROOT = matches[0]
print('Detected dataset root:', DATA_ROOT)
if DATASET == 'fashionIQ':
    f = Path(REPO)/'data/fashionIQ.py'
    txt = f.read_text(encoding='utf-8')
    txt = __import__('re').sub(r'(?m)^base_path = Path\(.+\)$', f'base_path = Path(r"{DATA_ROOT}")', txt, count=1)
    f.write_text(txt, encoding='utf-8')
else:
    f = Path(REPO)/'data/cirr.py'
    txt = f.read_text(encoding='utf-8')
    txt = __import__('re').sub(r'(?m)^base_path = Path\(.+\)$', f'base_path = Path(r"{DATA_ROOT}")', txt, count=1)
    f.write_text(txt, encoding='utf-8')
print('Dataset path configured')

In [ ]:
# Cell 6 — create an efficient Kaggle training config and start training
# train_qure.py evaluates each epoch and saves model.pth every epoch. Set a modest EPOCHS first to estimate total run time.
# Repair/check runtime dependencies in the exact interpreter used by the training subprocess.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'iopath==0.1.10', 'fairscale==0.4.4'], check=True)
subprocess.run([sys.executable, '-c', 'import iopath, fairscale; print("iopath and fairscale imports: OK")'], check=True)
from pathlib import Path
cfg_path = Path(REPO)/'kaggle_train.json'
base_cfg = json.loads((Path(REPO)/'configs'/DATASET/'train.json').read_text())
base_cfg.update({
    'epoch': EPOCHS, 'batch_size': BATCH_SIZE, 'num_workers': NUM_WORKERS,
    'device_idx': '0', 'experiment_dir': '/kaggle/working/qure_runs',
    # 'debug' avoids the repo's optional W&B logging block (which references an undefined topk).
    'experiment_description': 'debug', 'dataset': DATASET, 'mode': 'train',
    'use_temp': True, 'prompt_flow_kl_weight': 0.01,
})
if RESUME_CHECKPOINT:
    assert os.path.isfile(RESUME_CHECKPOINT), RESUME_CHECKPOINT
    base_cfg['checkpoint_path'] = RESUME_CHECKPOINT
cfg_path.write_text(json.dumps(base_cfg, indent=2))
print(json.dumps(base_cfg, indent=2))
start = time.time()
subprocess.run([sys.executable, 'train_qure.py', '--config_path='+str(cfg_path)], cwd=REPO, check=True)
print(f'Training wall time: {(time.time()-start)/3600:.2f} hours')

In [ ]:
# Cell 7 — find the latest saved model and run validation/evaluation
from pathlib import Path
run_dirs = sorted([p for p in Path('/kaggle/working/qure_runs').glob('debug_*') if (p/'model.pth').is_file()], key=lambda p: p.stat().st_mtime)
assert run_dirs, 'No run directory/checkpoint found'
RUN_DIR = run_dirs[-1]
CHECKPOINT = RUN_DIR/'model.pth'
assert CHECKPOINT.is_file(), CHECKPOINT
print('Evaluating checkpoint:', CHECKPOINT)
if EVAL_AFTER_TRAIN:
    eval_cfg = json.loads((Path(REPO)/'configs'/DATASET/'eval.json').read_text())
    eval_cfg.update({'dataset': DATASET, 'mode': 'eval', 'device_idx': '0',
                     'batch_size': BATCH_SIZE, 'num_workers': NUM_WORKERS,
                     'model': base_cfg['model'], 'backbone': base_cfg['backbone'],
                     'blip_size': base_cfg['blip_size'], 'img_size': base_cfg['img_size'],
                     'text_encoder': base_cfg['text_encoder'], 'use_temp': base_cfg['use_temp'],
                     'experiment_dir': '/kaggle/working/qure_eval_runs',
                     'experiment_description': 'debug',
                     'pretrained_path': str(RUN_DIR)})
    eval_path = Path(REPO)/'kaggle_eval.json'
    eval_path.write_text(json.dumps(eval_cfg, indent=2))
    eval_script = 'evaluate_qure/evaluate_fiq.py' if DATASET == 'fashionIQ' else 'evaluate_qure/evaluate_cirr.py'
    subprocess.run([sys.executable, eval_script, '--config_path='+str(eval_path)], cwd=REPO, check=True)
print('Checkpoint and logs:', RUN_DIR)

In [ ]:
# Cell 8 — package all results into one downloadable zip in /kaggle/working
# Kaggle persists /kaggle/working when you use Save Version and include notebook output.
from pathlib import Path
RESULT_BUNDLE = Path('/kaggle/working/qure_result_bundle')
if RESULT_BUNDLE.exists(): shutil.rmtree(RESULT_BUNDLE)
RESULT_BUNDLE.mkdir(parents=True)
shutil.copytree(RUN_DIR, RESULT_BUNDLE/RUN_DIR.name)
for eval_dir in [Path(REPO)/'cir_eval/fiq', Path(REPO)/'cir_eval/cirr/submission']:
    if eval_dir.exists(): shutil.copytree(eval_dir, RESULT_BUNDLE/eval_dir.name, dirs_exist_ok=True)
result_zip = shutil.make_archive('/kaggle/working/QuRe_'+DATASET+'_results', 'zip', root_dir=str(RESULT_BUNDLE))
print('Download this from the notebook Output tab:', result_zip)
print('Checkpoint:', CHECKPOINT)
if DATASET == 'fashionIQ':
    print('Eval JSONs:', list(Path(REPO, 'cir_eval/fiq').glob('*.json')))
else:
    print('CIRR validation metrics were printed during training. This eval script writes test1 submission predictions (ground-truth metrics are unavailable for test1).')